# Feature sharing: replay epoch 1 tail27

Attach the four existing Datasets, select GPU T4 x2, enable Internet and Save & Run All. The probe rebuilds the sampler from metadata and reads only the target H5 rows. Send csa_sharing_tail27_review.zip. The result compares ordinary and shared FP32 gradients per rank and after DDP, and records ReLU sign changes for neuron 933. No optimizer updates.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil,tarfile
from pathlib import Path
REVISION='423fac4e244ce125abe4f30b99353a72852e4087'
MAPPING={'experiments/clip_patch_baseline/full_data.py': 'full_data.py', 'experiments/clip_patch_baseline/full_runtime.py': 'full_runtime.py', 'experiments/clip_patch_baseline/run_full.py': 'run_full.py', 'experiments/clip_patch_baseline/patch_model.py': 'patch_model.py', 'experiments/clip_patch_baseline/patch_beam.py': 'patch_beam.py', 'experiments/clip_patch_baseline/score_full.py': 'score_full.py', 'experiments/clip_patch_baseline/patch_data.py': 'patch_data.py', 'experiments/clip_patch_baseline/test_full_data.py': 'test_full_data.py', 'experiments/clip_patch_baseline/make_full_notebook.py': 'make_full_notebook.py', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': 'SMOKE_SOURCE.json', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'FULL_XE10_PROTOCOL.md', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/cuda_runtime_guard.py': 'cuda_runtime_guard.py', 'experiments/csa_paper/saved_quality_data.py': 'saved_quality_data.py', 'experiments/clip_patch_baseline/local_stage.py': 'local_stage.py', 'experiments/clip_patch_baseline/test_local_stage.py': 'test_local_stage.py', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': 'LOCAL_STAGE_V3_PROTOCOL.md', 'experiments/clip_patch_baseline/semantic_data.py': 'semantic_data.py', 'experiments/clip_patch_baseline/semantic_patch_model.py': 'semantic_patch_model.py', 'experiments/clip_patch_baseline/semantic_training.py': 'semantic_training.py', 'experiments/clip_patch_baseline/test_semantic_data.py': 'test_semantic_data.py', 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md': 'BANK_PROMPT_SCRATCH_PROTOCOL.md', 'experiments/csa_paper/model.py': 'bank_layers.py', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': 'SAVE_BANK_V130_PACKAGE.json', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'bank_words_corrected.json', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'experiments/clip_patch_baseline/memory_fusion.py': 'memory_fusion.py', 'experiments/clip_patch_baseline/csa_full_data.py': 'csa_full_data.py', 'experiments/clip_patch_baseline/csa_full_runtime.py': 'csa_full_runtime.py', 'experiments/clip_patch_baseline/run_csa_full.py': 'run_csa_full.py', 'experiments/clip_patch_baseline/score_csa_full.py': 'score_csa_full.py', 'experiments/clip_patch_baseline/test_csa_full_data.py': 'test_csa_full_data.py', 'experiments/clip_patch_baseline/csa_patch_model.py': 'csa_patch_model.py', 'experiments/clip_patch_baseline/csa_shared_model.py': 'csa_shared_model.py', 'experiments/clip_patch_baseline/csa_sharing_data.py': 'csa_sharing_data.py', 'experiments/clip_patch_baseline/gradient_agreement.py': 'gradient_agreement.py', 'experiments/clip_patch_baseline/test_gradient_agreement.py': 'test_gradient_agreement.py', 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md': 'CSA_SHARED_FULL_XE10_PROTOCOL.md', 'experiments/clip_patch_baseline/make_csa_full_notebook.py': 'make_csa_full_notebook.py'}
HASHES={'experiments/clip_patch_baseline/full_data.py': 'ab69cf61d4ad0456d9a869929750ab10f3a2a33b6c76683689539e57e4015a74', 'experiments/clip_patch_baseline/full_runtime.py': '230f7e10d96005fbffc33fb151a7a1d7c7245326e73826134a400772dadd04bb', 'experiments/clip_patch_baseline/run_full.py': '32dfb92228fad22d4f503680db1c8226db1e6abf9d7dd253786c3f16be9a97c0', 'experiments/clip_patch_baseline/patch_model.py': 'dfe6b1d83bfc1e3c7961268fb104bc1f4092129e7d9400841568a3204dc57a70', 'experiments/clip_patch_baseline/patch_beam.py': '53fcb5b22bb9f05d795fbd4f539cfd4ff0af39c1dea679f7df1a933edf2754be', 'experiments/clip_patch_baseline/score_full.py': 'fed170a6d12d1ab1bb969ae7eac29ee0855059417f6e2e876da193e279ca819a', 'experiments/clip_patch_baseline/patch_data.py': '3706ffec9c128c8fe12c4f21ba071939268fdd526ed614034cca8dfec55674d5', 'experiments/clip_patch_baseline/test_full_data.py': 'f47160cf9cd85c96a4495dfa8264c0cd5e7cd60e8d5b6a77022932399e557431', 'experiments/clip_patch_baseline/make_full_notebook.py': '2417f0c691e291808b9f3b44f8bdb60090fcaf8bb487d71948538a7d61d5679d', 'experiments/clip_patch_baseline/SMOKE_SOURCE.json': '6413058df1575ae8266c6f7ec29eea3cb60917147126bc55961de47c949abe49', 'experiments/clip_patch_baseline/FULL_XE10_PROTOCOL.md': 'a7c482bfe9efe3f23d4ebf235872204b8e0e0e9e69fae71e3a7b04b8980a986f', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/cuda_runtime_guard.py': '640b5debc81607e3504930e4042571636cc3fbd525a6d0e5eb45abdd141fb773', 'experiments/csa_paper/saved_quality_data.py': 'f8a092506e3adf96292203190ea2f83de397d3028a3ddc6293d58fd360199a1e', 'experiments/clip_patch_baseline/local_stage.py': '76ec1d4ace24fe2d95ea396d489692b7ff3c6289b71fd88c81b110cd2ad9426b', 'experiments/clip_patch_baseline/test_local_stage.py': '028e8bff388c44123368dac2aa21887d589adf1e02bb2d3f7082870564b9cfbb', 'experiments/clip_patch_baseline/LOCAL_STAGE_V3_PROTOCOL.md': '83ba0fd376a2f1f6069eca1a082c954e8316b236fa343807f813a40eb0a9487d', 'experiments/clip_patch_baseline/semantic_data.py': 'a57af931f9e6be85f6efeee668b09fcca14c8a3bd54ebd695d9014ede2ab0540', 'experiments/clip_patch_baseline/semantic_patch_model.py': '10a385859ea1389a71f14a7a76e16a6989feb73fd581ca89659eab50ec53a0f3', 'experiments/clip_patch_baseline/semantic_training.py': '001f05f3e6959570571135a55abb7bc740217a447371bab627af8c72e0f26d11', 'experiments/clip_patch_baseline/test_semantic_data.py': 'e9d9132db2e83b47e7f29cd1c465389cb4f16db2cd62a1cf27f048a06e29a572', 'experiments/clip_patch_baseline/BANK_PROMPT_SCRATCH_PROTOCOL.md': '93d1ebef1d359d393323951a152f50d16559c0fd235dac27eda4940653399a0c', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'experiments/clip_patch_baseline/memory_fusion.py': 'a70947e22263ddd27a7896a94cfb35cfff73102370288c53f86529964ea8f087', 'experiments/clip_patch_baseline/csa_full_data.py': '4028cd2b656d2244cabe2b4ab510b7a701bd8d462157adce62e10bf2060ca8c1', 'experiments/clip_patch_baseline/csa_full_runtime.py': '508b10b38871041a7f4964d73291a82f55b7dae42a3e6651b82e1a451fff1325', 'experiments/clip_patch_baseline/run_csa_full.py': '3c5639e29b8a5137c3e677ae110d87cb697288e93bdbc8826ccd009ffd4ff7bd', 'experiments/clip_patch_baseline/score_csa_full.py': '25d912e57a96be0ac2cdef102d681ebff2afb00521de4e9ba7b2eea208d8bcb8', 'experiments/clip_patch_baseline/test_csa_full_data.py': '4f3f5e84ec751a54fe2a338441c9c6644398cbac950b4001c015b0a959e63b07', 'experiments/clip_patch_baseline/csa_patch_model.py': '99b6441c4205b142256192cc3d840a905e00ecc1e730f30e1f5eb2c3cf8dc220', 'experiments/clip_patch_baseline/csa_shared_model.py': '9599759a16e8570fbb625c88c821f2cf26776be8a317db650adcd34ef563cbf8', 'experiments/clip_patch_baseline/csa_sharing_data.py': '4d89428b8f833e34522bd51ec25640015edbebae4f0b5b81450058e2ebcb1243', 'experiments/clip_patch_baseline/gradient_agreement.py': 'dc06ffd922fccd9e14e860d07644dddcf320f7d9ac6410e3166cc4f53200971c', 'experiments/clip_patch_baseline/test_gradient_agreement.py': '2b2aa46091a5a752b3eab573f701c0f1e2849349d829e1c4d6a4d99ae73285a6', 'experiments/clip_patch_baseline/CSA_SHARED_FULL_XE10_PROTOCOL.md': 'c9f4dd7b6507218cc63cd6c72809d2cf97670ce0b1172c6bb5159ef6d25b604a', 'experiments/clip_patch_baseline/make_csa_full_notebook.py': 'c9f8e3e793b0841df24e938103cbf8a83396a297ded1634108afec10e2cead9b'}
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
RESUME_ROOT='' # Optional preserved full-run output from THIS exact recipe. Blank starts fresh.
WALL_HOURS=0.5 # Parent stop cap, not ETA/billed quota. Do not auto-extend.
OUTPUT=Path('/kaggle/working/csa_sharing_tail27')
for p in (VISUAL_CACHE,COCO_JSON,BANK_ROOT,PROMPT_ROOT):
    if not p.exists():raise FileNotFoundError('Attach/correct inputs BEFORE GPU: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve previous output; use a fresh runtime/output path')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='clip-patch-full10-',dir='/tmp'));DEPS=SOURCE/'deps';LOG=OUTPUT/'notebook.log'
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json';CHAIR=SOURCE/'chair'
cfg=dict(csa_shared_full=True,semantic=True,bank_root=str(BANK_ROOT),prompt_root=str(PROMPT_ROOT),feature_storage='local_fp16_memmap_v2',prepare_seconds=5400,output=str(OUTPUT),stage=str(SOURCE/'stage'),dataset_json=str(COCO_JSON),visual_cache=str(VISUAL_CACHE),worker_seconds=600,resume_root=RESUME_ROOT,chair=str(CHAIR))
def write_config():
    CONFIG.write_text(json.dumps(cfg,indent=2));(OUTPUT/'protocol.json').write_text(json.dumps(cfg,indent=2))
write_config()
def download(url,expected,path):
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent cap during download')
    raw=urllib.request.urlopen(url,timeout=min(120,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned asset: '+path.name)
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)

ACCEL_DEPS=SOURCE/'accelerate-deps'
os.environ['PYTHONPATH']=str(ACCEL_DEPS)+os.pathsep+str(SOURCE)


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent cap exceeded')
    timeout=min(timeout,PARENT_DEADLINE-time.monotonic())
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Tail27 probe active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Phase/parent cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    for name,destination in MAPPING.items():download('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+name,HASHES[name],SOURCE/destination)
    (OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2))
    run([sys.executable,'-u',SOURCE/'test_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_local_stage.py'],120)
    run([sys.executable,'-u',SOURCE/'test_semantic_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_csa_full_data.py'],120)
    run([sys.executable,'-u',SOURCE/'test_gradient_agreement.py'],120)
    run([sys.executable,'-u',SOURCE/'cuda_runtime_guard.py','--output',OUTPUT],60)

    run([sys.executable,'-m','pip','download','-q','--no-deps','--only-binary=:all:','--dest',SOURCE,'accelerate==1.15.0'],180)
    wheel=SOURCE/'accelerate-1.15.0-py3-none-any.whl'
    if hashlib.sha256(wheel.read_bytes()).hexdigest()!='97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec':raise ValueError('Changed Accelerate wheel')
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',ACCEL_DEPS,wheel],180)
    run([sys.executable,'-c',"import accelerate;assert accelerate.__version__=='1.15.0';print('PinnedAccelerate',accelerate.__version__,flush=True)"],120)
    run([sys.executable,'-c',"from csa_shared_model import SharedCSAPatchCaptioner;from csa_full_runtime import make_model;print('Neural module imports PASS; no model instantiated',flush=True)"],120)
    (OUTPUT/'accelerate_install.json').write_text(json.dumps(dict(version='1.15.0',wheel_sha256=hashlib.sha256(wheel.read_bytes()).hexdigest(),dependencies_reinstalled=False)))

    script=SOURCE/'probe_sharing_tail.py'
    script.write_text('"""User-run exact tail27 probe. Only selected H5 feature rows are read."""\nimport json\nimport sys\nfrom datetime import timedelta\nfrom pathlib import Path\n\nimport numpy as np\nfrom csa_full_data import order_for_epoch\nfrom csa_sharing_data import image_plan, local_events\nfrom full_data import FeatureReader, load_stage, prepare\nfrom gradient_agreement import gradient_agreement\nfrom real_bank_data import read, save\nfrom semantic_data import SemanticContext, checked_inputs, record_arrays\n\n\ndef stage_tail(cfg):\n    prepare(cfg)\n    stage = Path(cfg[\'stage\'])\n    data, _, _ = load_stage(stage)\n    events = order_for_epoch(data, 1)[-27:]\n    indices, inverse = np.unique(data[\'caption_image\'][events], return_inverse=True)\n    if sum(len(image_plan(data[\'caption_image\'], local_events(events, rank), True)[0])\n           for rank in range(2)) != 7:\n        raise ValueError(\'Tail encoder count differs from V2\')\n    ids = [int(x) for x in data[\'ids\'][indices]]\n    _, root, cohort, texts, vectors, _, banks, _ = checked_inputs(cfg)\n    if not set(ids) <= set(cohort[\'train\']):\n        raise ValueError(\'Tail contains a non-TRAIN image\')\n    selected = {}\n    wanted = set(ids)\n    with (root/\'prompt_links.jsonl\').open(encoding=\'utf8\') as stream:\n        for line in stream:\n            row = json.loads(line)\n            coco = row[\'coco_id\']\n            if coco in wanted:\n                if coco in selected or row[\'group\'] != \'train\':\n                    raise ValueError(\'Wrong tail prompt provenance\')\n                selected[coco] = record_arrays(row, texts)\n    if set(selected) != wanted:\n        raise ValueError(\'Missing tail prompts\')\n    for name in selected[ids[0]]:\n        np.save(stage/f\'sem_{name}.npy\', np.stack([selected[i][name] for i in ids]))\n    np.save(stage/\'sem_text_vectors.npy\', vectors)\n    for name, bank in banks.items():\n        np.save(stage/f\'sem_{name}_bank.npy\', bank)\n    reader = FeatureReader(stage)\n    try:\n        patches = np.stack([reader.get(i) for i in indices])\n    finally:\n        reader.close()\n    sequences = data[\'sequences\'][events]\n    if int((sequences[:, 1:] != 0).sum()) != 303:\n        raise ValueError(\'Tail token count differs from V2\')\n    np.savez(stage/\'tail.npz\', patches=patches, sequences=sequences, image=inverse)\n    save(Path(cfg[\'output\'])/\'tail_inputs.json\',\n         dict(coco_ids=ids, event_ids=events.tolist(), feature_rows=len(ids), global_tokens=303))\n\n\ndef probe(cfg):\n    import torch\n    import torch.distributed as dist\n    from accelerate import Accelerator\n    from accelerate.utils import DistributedDataParallelKwargs, InitProcessGroupKwargs\n    from csa_shared_model import SharedCSAPatchCaptioner\n    from full_runtime import tensor_digest\n\n    torch.set_num_threads(1)\n    accel = Accelerator(mixed_precision=\'no\', gradient_accumulation_steps=1,\n                        dynamo_backend=\'no\', kwargs_handlers=[\n                            DistributedDataParallelKwargs(broadcast_buffers=False),\n                            InitProcessGroupKwargs(backend=\'nccl\', timeout=timedelta(seconds=180))])\n    if accel.num_processes != 2 or accel.device.type != \'cuda\':\n        raise ValueError(\'Select GPU T4 x2\')\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.manual_seed(42)\n    stage, out = Path(cfg[\'stage\']), Path(cfg[\'output\'])\n    words = read(stage/\'vocabulary.json\')\n    banks = [torch.from_numpy(np.load(stage/f\'sem_{kind}_bank.npy\', allow_pickle=False))\n             for kind in (\'noun\', \'verb\')]\n    model = SharedCSAPatchCaptioner(len(words), *banks).to(accel.device).eval()\n    wrapped = accel.prepare_model(model)\n    rank = accel.process_index\n    with np.load(stage/\'tail.npz\', allow_pickle=False) as tail:\n        rows = local_events(np.arange(27), rank)\n        image = tail[\'image\'][rows]\n        patches = tail[\'patches\'].astype(np.float32)\n        seq = tail[\'sequences\'][rows].astype(np.int64)\n    seq = torch.from_numpy(seq[:, :int((seq != 0).sum(1).max())]).to(accel.device)\n    context = SemanticContext(stage)\n    activation = []\n    hook = model.core.decoder[1].ffn.net[0].register_forward_hook(\n        lambda module, inputs, output: activation.append(output[..., 933].detach().cpu().numpy()))\n    targets = (\'core.decoder.1.ffn.net.0.weight\', \'core.decoder.1.ffn.net.0.bias\')\n\n    def run(shared, synchronized):\n        first, inverse = image_plan(image, np.arange(len(image)), shared)\n        selected = image[first]\n        x = torch.from_numpy(patches[selected]).to(accel.device)\n        p = context.images(selected, accel.device)\n        inverse = None if inverse is None else torch.from_numpy(inverse).to(accel.device)\n        wrapped.zero_grad(set_to_none=True)\n        net = wrapped if synchronized else model\n        logits = net(x, seq[:, :-1], p, inverse)\n        nll = torch.nn.functional.cross_entropy(logits.flatten(0, 1), seq[:, 1:].flatten(),\n                                                ignore_index=0, reduction=\'sum\')\n        accel.backward(nll * (2 / 303))\n        names = None if synchronized else targets\n        grads = {name: p.grad.detach().cpu().numpy().copy() for name, p in model.named_parameters()\n                 if names is None or name in names}\n        return float(nll.detach()), grads\n\n    ordinary_loss, ordinary_local = run(False, False)\n    shared_loss, shared_local = run(True, False)\n    hook.remove()\n    ordinary, shared = activation\n    flips = (ordinary > 0) != (shared > 0)\n    valid = seq[:, 1:].ne(0).cpu().numpy()\n    locations = np.argwhere(flips)\n    sign_changes = [dict(event_index=int(rows[i]), token=int(j), valid_target=bool(valid[i, j]),\n                         ordinary=float(ordinary[i, j]), shared=float(shared[i, j]))\n                    for i, j in locations]\n    save(out/f\'tail_rank{rank}.json\',\n         dict(torch_version=torch.__version__, gpu=torch.cuda.get_device_name(accel.device),\n              initial_tensor_sha256=tensor_digest(model), caption_events=len(rows),\n              ordinary_nll_sum=ordinary_loss, shared_nll_sum=shared_loss,\n              neuron_933_sign_changes=sign_changes,\n              local_gradients={n:gradient_agreement(shared_local[n], ordinary_local[n]) for n in targets}))\n    ordinary_nll, ordinary_ddp = run(False, True)\n    shared_nll, shared_ddp = run(True, True)\n    losses = torch.tensor([ordinary_nll, shared_nll], dtype=torch.float64, device=accel.device)\n    dist.all_reduce(losses)\n    if rank == 0:\n        diagnostics = {n:gradient_agreement(shared_ddp[n], g) for n, g in ordinary_ddp.items()}\n        nll = (losses / 303).tolist()\n        failed = [n for n, v in diagnostics.items() if not v[\'passed\']]\n        save(out/\'tail_ddp.json\', dict(passed=not failed and abs(nll[0]-nll[1]) <= 5e-5,\n             ordinary_nll=nll[0], shared_nll=nll[1], failed_parameters=failed,\n             gradient_diagnostics=diagnostics, optimizer_updates=0))\n    accel.end_training()\n\n\nif __name__ == \'__main__\':\n    {\'prepare\': stage_tail, \'probe\': probe}[sys.argv[1]](read(sys.argv[2]))\n',encoding='utf8')
    (OUTPUT/'probe_source_sha256.txt').write_text(hashlib.sha256(script.read_bytes()).hexdigest())
    run([sys.executable,'-u',script,'prepare',CONFIG],900)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',script,'probe',CONFIG],300)
    complete=all((OUTPUT/name).is_file() for name in ('tail_rank0.json','tail_rank1.json','tail_ddp.json'))
    if not complete:raise RuntimeError('Missing tail probe report')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc())
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,optimizer_updates=0)))
    review=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for path in OUTPUT.iterdir():
            if path.is_file():z.write(path,path.name)
    print('SEND REVIEW:',review,flush=True)
